<a href="https://colab.research.google.com/github/dynamodenis/QDrant/blob/main/vector_search/movie_recomendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semantic Movie Search
## Project Overview: When Search Understands Meaning

Imagine asking a search engine: “Show me movies about questioning reality and the nature of existence” and getting back The Matrix, Inception, and Ex Machina, but not because these titles contain those exact words, but because the system understands what these films are actually about.

That’s semantic search. And you’re about to build one.

We’ll take detailed movie descriptions and apply the chunking strategies you learned earlier, embed those chunks using sentence transformers, and store them in Qdrant with rich metadata. The result is a search engine that understands themes, moods, and concepts.

This project synthesizes everything from today: points and vectors, distance metrics, payloads, chunking strategies, and embedding models. By the end, you’ll have a working system that can find movies by plot, theme, or emotional resonance.

## Understanding the Challenge

Our dataset consists of 13 science fiction movies with detailed, literary descriptions. Here’s the challenge: each description contains 240-460 tokens, but our embedding model (all-MiniLM-L6-v2) can only embed 256 tokens or less.

This is where chunking becomes essential.

```
# Example: A movie description that's too long for our embedding model
movie_example = {
    "name": "Ex Machina",
    "year": 2014,
    "description": """Alex Garland's Ex Machina is a cerebral, slow-burning psychological
    thriller that delves into the ethics and consequences of artificial intelligence.
    The story begins with Caleb, a young programmer at a tech conglomerate, who wins
    a contest to spend a week at the secluded estate of Nathan, the reclusive CEO..."""
    # This continues for 386 tokens - too long for optimal embedding!
}

```



## The Three-Vector Experiment

Here’s what makes this demo unique: we’ll create three different vector spaces in a single collection, each representing a different chunking strategy. This lets us directly compare how chunking affects search quality.

Side note: Creating three different vector spaces in a single collection is almost as expensive as having one collection per vector space. We do it here purely for comparison convenience.

In [ ]:
!pip install qdrant-client

!pip install llama-index llama-index-embeddings-huggingface

In [ ]:
from sentence_transformers import SentenceTransformer # Embedding modal
from qdrant_client import QdrantClient, models # Vector database

# Initilize components and load the pretrained sentence modal from huggingface, its small and efficient
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# In memory training since its small and its for test. NO HNSW built -> queries are a full scan.
client = QdrantClient(":memory:")

# For ANN/HNSW:
# client = QdrantClient(url="http://localhost:6333")

In [2]:
# Create collection with three NAMED vectors

if not client.collection_exists("movie_search"):
  client.create_collection(
      collection_name="movie_search",
      vectors_config={
          "fixed": models.VectorParams(size=384, distance=models.Distance.COSINE),
          "sentence": models.VectorParams(size=384, distance=models.Distance.COSINE),
          "semantic": models.VectorParams(size=384, distance=models.Distance.COSINE),
      }
  )
  print("Collection created")
else:
    print("Collection already exists")


Collection created


Each vector space will store the same movie content, chunked differently:

1. Fixed: Raw 40-token chunks (may break mid-sentence)
2. Sentence: Sentence-aware chunks with overlap
3. Semantic: Meaning-aware chunks using embedding similarity

## Implementing the Chunking Strategies

**The key difference**: Fixed chunking may split mid-sentence. Sentence chunking respects grammar. Semantic chunking respects meaning.

Here’s where the chunking concepts from earlier lessons come alive. We’ll implement three different approaches and see how they perform:



In [3]:
from transformers import AutoTokenizer
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
MAX_TOKENS = 40

def fixed_size_chunks(text, size=MAX_TOKENS):
    """Fixed-size chunking: splits at exact token boundaries"""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    return [
        tokenizer.decode(tokens[i:i+size], skip_special_tokens=True)
        for i in range(0, len(tokens), size)
    ]

def sentence_chunks(text):
    """Sentence-aware chunking: respects sentence boundaries"""
    splitter = SentenceSplitter(chunk_size=MAX_TOKENS, chunk_overlap=10)
    return splitter.split_text(text)

def semantic_chunks(text):
    """Semantic chunking: uses embedding similarity to find natural breaks.
    Note: still constrained by the embed model's context window (same as retrievers)."""
    from llama_index.core import Document

    semantic_splitter = SemanticSplitterNodeParser(
        buffer_size=1,
        breakpoint_percentile_threshold=95,
        embed_model=HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
    )
    nodes = semantic_splitter.get_nodes_from_documents([Document(text=text)])
    return [node.text for node in nodes]


## Processing and Uploading the Data

For each movie description, we apply all three chunking strategies, embed the resulting chunks, and store them with their respective vector names:

In [ ]:
points = []
idx = 0
movies_data = [
    {
        "name": "Inception",
        "description": "A thief who steals corporate secrets through the use of dream-sharing technology is given the inverse task of planting an idea into the mind of a C.E.O. The film explores the layers of reality and the subconscious mind. Dom Cobb must navigate complex dream architectures while dealing with his own repressed memories.",
        "author": "Christopher Nolan",
        "year": 2010
    },
    {
        "name": "The Matrix",
        "description": "A computer hacker learns from mysterious rebels about the true nature of his reality and his role in the war against its controllers. Humans are trapped in a simulated reality created by sentient machines. Neo must decide whether to take the red pill and face the harsh truth of the real world.",
        "author": "Lana and Lilly Wachowski",
        "year": 1999
    },
    {
        "name": "Interstellar",
        "description": "When Earth becomes uninhabitable in the future, a farmer and ex-NASA pilot, Joseph Cooper, is tasked to pilot a spacecraft, along with a team of researchers, to find a new planet for humans. It features themes of love transcending time and space, gravity's effect on time, and the survival of the human race.",
        "author": "Christopher Nolan",
        "year": 2014
    },
    {
        "name": "Parasite",
        "description": "Greed and class discrimination threaten the newly formed symbiotic relationship between the wealthy Park family and the destitute Kim clan. A dark comedy thriller that explores social inequality in South Korea. The basement of the Park home hides a secret that will change both families forever.",
        "author": "Bong Joon-ho",
        "year": 2019
    },
    {
        "name": "The Godfather",
        "description": "The aging patriarch of an organized crime dynasty transfers control of his clandestine empire to his reluctant son. Set in post-WWII New York, the story follows the transformation of Michael Corleone from a war hero to a ruthless mafia boss. It is a tale of family loyalty and the price of power.",
        "author": "Francis Ford Coppola",
        "year": 1972
    },
    {
        "name": "Spirited Away",
        "description": "During her family's move to the suburbs, a sullen 10-year-old girl wanders into a world ruled by gods, witches, and spirits, and where humans are changed into beasts. Chihiro must work in a magical bathhouse to save her parents. The film is a masterpiece of Japanese animation and traditional folklore.",
        "author": "Hayao Miyazaki",
        "year": 2001
    },
    {
        "name": "Blade Runner 2049",
        "description": "Young Blade Runner K's discovery of a long-buried secret leads him to track down former Blade Runner Rick Deckard, who's been missing for thirty years. The story deals with the soul of replicants and what it means to be human in a desolate future. Visuals are characterized by heavy fog and neon landscapes.",
        "author": "Denis Villeneuve",
        "year": 2017
    },
    {
        "name": "Arrival",
        "description": "A linguist works with the military to communicate with alien lifecorns after twelve mysterious spacecraft appear around the world. As Louise Banks learns the alien language, she begins to perceive time in a non-linear fashion. The film is a deep dive into communication, grief, and deterministic fate.",
        "author": "Denis Villeneuve",
        "year": 2016
    },
    {
        "name": "The Dark Knight",
        "description": "When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice. The Joker challenges Batman's moral code. Gotham City becomes a battleground for the soul of its citizens.",
        "author": "Christopher Nolan",
        "year": 2008
    },
    {
        "name": "Everything Everywhere All At Once",
        "description": "A middle-aged Chinese immigrant is swept up into an insane adventure in which she alone can save existence by exploring other universes and connecting with the lives she could have led. It blends martial arts action with family drama and existential philosophy. Taxes and laundry become the center of a multiverse war.",
        "author": "Daniel Kwan and Daniel Scheinert",
        "year": 2022
    },
    {
        "name": "Pulp Fiction",
        "description": "The lives of two mob hitmen, a boxer, a gangster and his wife, and a pair of diner bandits intertwine in four tales of violence and redemption. The narrative is non-linear and famous for its stylized dialogue. Los Angeles serves as the backdrop for these gritty and darkly humorous stories.",
        "author": "Quentin Tarantino",
        "year": 1994
    },
    {
        "name": "Children of Men",
        "description": "In 2027, in a chaotic world in which women have become somehow infertile, a former activist agrees to help transport a miraculously pregnant woman to a sanctuary at sea. The film is known for its long, immersive takes and bleak social commentary. Hope is a rare commodity in this collapsing society.",
        "author": "Alfonso Cuarón",
        "year": 2006
    },
    {
        "name": "Dune: Part Two",
        "description": "Paul Atreides unites with Chani and the Fremen while on a warpath of revenge against the conspirators who destroyed his family. He must choose between the love of his life and the fate of the known universe. The desert planet Arrakis is the stage for a massive religious and political conflict.",
        "author": "Denis Villeneuve",
        "year": 2024
    },
    {
        "name": "Seven",
        "description": "Two detectives, a rookie and a veteran, hunt a serial killer who uses the seven deadly sins as his motives. The city is perpetually raining and dark, reflecting the grim nature of the investigation. The final act contains one of the most famous and shocking twists in cinema history.",
        "author": "David Fincher",
        "year": 1995
    },
    {
        "name": "The Shawshank Redemption",
        "description": "Over the course of several years, two convicts form a friendship, seeking consolation and, eventually, redemption through basic compassion. Andy Dufresne is wrongly convicted of murder but never loses hope. The film examines the effects of institutionalization and the endurance of the human spirit.",
        "author": "Frank Darabont",
        "year": 1994
    }
]
for movie in movies_data:  # Process each movie
    # Fixed-size chunks
    for chunk in fixed_size_chunks(movie["description"]):
        points.append(models.PointStruct(
            id=idx,
            vector={"fixed": encoder.encode(chunk).tolist()},
            payload={**movie, "chunk": chunk, "chunking": "fixed"}
        ))
        idx += 1

    # Sentence-aware chunks
    for chunk in sentence_chunks(movie["description"]):
        points.append(models.PointStruct(
            id=idx,
            vector={"sentence": encoder.encode(chunk).tolist()},
            payload={**movie, "chunk": chunk, "chunking": "sentence"}
        ))
        idx += 1

    # Semantic chunks
    for chunk in semantic_chunks(movie["description"]):
        points.append(models.PointStruct(
            id=idx,
            vector={"semantic": encoder.encode(chunk).tolist()},
            payload={**movie, "chunk": chunk, "chunking": "semantic"}
        ))
        idx += 1

client.upload_points(collection_name='movie_search', points=points)
print(f"Uploaded {idx} vectors across three chunking strategies")


## Comparing Search Results

Now comes the fascinating part: testing how different chunking strategies affect search quality. Let’s create a helper function to compare results:

In [7]:
def search_and_compare(query, k=3):
    """Compare search results across all three chunking strategies"""
    print(f"Query: '{query}'\n")

    for strategy in ['fixed', 'sentence', 'semantic']:
        results = client.query_points(
            collection_name='movie_search',
            query=encoder.encode(query).tolist(),
            using=strategy,
            limit=k,
        )

        print(f"--- {strategy.upper()} CHUNKING ---")
        for i, point in enumerate(results.points, 1):
            payload = point.payload
            print(f"{i}. {payload['name']} ({payload['year']}) | Score: {point.score:.3f}")
            print(f"   Chunk: {payload['chunk'][:100]}...")
        print()

# Test with different queries
search_and_compare("alien invasion")
# search_and_compare("questioning reality and existence")


Query: 'alien invasion'

--- FIXED CHUNKING ---
1. Dune: Part Two (2024) | Score: 0.359
   Chunk: life and the fate of the known universe. the desert planet arrakis is the stage for a massive religi...
2. Interstellar (2014) | Score: 0.349
   Chunk: new planet for humans. it features themes of love transcending time and space, gravity ' s effect on...
3. Arrival (2016) | Score: 0.318
   Chunk: a linguist works with the military to communicate with alien lifecorns after twelve mysterious space...

--- SENTENCE CHUNKING ---
1. Arrival (2016) | Score: 0.357
   Chunk: As Louise Banks learns the alien language, she begins to perceive time in a non-linear fashion. The ...
2. Arrival (2016) | Score: 0.353
   Chunk: A linguist works with the military to communicate with alien lifecorns after twelve mysterious space...
3. Interstellar (2014) | Score: 0.330
   Chunk: along with a team of researchers, to find a new planet for humans....

--- SEMANTIC CHUNKING ---
1. Interstellar (2014) | Score: 0

## Filtering by Metadata
Combine semantic search with traditional filters:

In [13]:
# Find Movies about AI made after 2000
results = client.query_points(
    collection_name='movie_search',
    query=encoder.encode("artificial intelligence").tolist(),
    using="semantic",
    query_filter=models.Filter(
        must=[models.FieldCondition(key="year", range=models.Range(gte=2000))]
    ),
    limit=4,
)

for point in results.points:
  print(f"{point.payload['name']} ({point.payload['year']}) | Score: {point.score:.3f}")

Inception (2010) | Score: 0.285
Arrival (2016) | Score: 0.278
Arrival (2016) | Score: 0.270
Inception (2010) | Score: 0.195


## Grouping Results to Avoid Duplicates

When multiple chunks from the same movie match, group results by movie title:

In [14]:
# Group by movie name to get unique recommendations
response = client.query_points_groups(
    collection_name="movie_search",
    query=encoder.encode("time travel and family relationships").tolist(),
    using="semantic",
    group_by="name",       # Group by movie title
    limit=3,               # Number of unique movies
    group_size=1,
)
for group in response.groups:
    print(f"{group.id} | Best match score: {group.hits[0].score:.3f}")

Parasite | Best match score: 0.335
The Godfather | Best match score: 0.311
Arrival | Best match score: 0.286


## What I have done

This summary brings together every concept from Day 1:

Chunking in Action: You’ve seen how different chunking strategies affect search quality. Fixed chunking is fast but crude, sentence chunking preserves readability, and semantic chunking captures meaning - but at computational cost.

Embeddings and Distance: The all-MiniLM-L6-v2 model converts movie descriptions into 384-dimensional vectors. Cosine similarity finds movies with similar themes, even when they use completely different words.

Payloads and Filtering: Rich metadata enables hybrid queries: “Find movies about AI made after 2000.” This combines semantic understanding with traditional database filtering.